In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🌐 FAQ Bot for a Single Website

## What We're Building

A chatbot that can answer questions about **any website** by:
1. **Crawling** the website pages
2. **Chunking and embedding** the content
3. **Answering user questions** using RAG (Retrieval-Augmented Generation)

This is like building a "talk to this website" feature.

## Architecture

```
Website URL → Crawl pages → Extract text → Chunk → Embed → Vector DB
                                                              ↓
User Question → Embed → Similarity Search → Top chunks → LLM → Answer
```

## What's Different from the PDF Q&A?

| PDF Q&A | Website FAQ Bot |
|---------|----------------|
| Single file input | Multiple pages to crawl |
| Text extraction (PyPDF) | HTML parsing (BeautifulSoup) |
| Static content | Links between pages |
| Page numbers for citations | URLs for citations |

## Stack
- **BeautifulSoup** — HTML parsing and text extraction
- **LangChain** — RAG pipeline orchestration
- **ChromaDB** — vector store
- **Ollama** — local LLM and embeddings

## Step 1 — Install & Import Dependencies

In [ ]:
# Install if needed (uncomment)
# !pip install langchain langchain-ollama langchain-community chromadb beautifulsoup4 requests -q

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.schema import Document

print("All imports successful!")

## Step 2 — Build the Web Crawler

We need a crawler that:
1. Starts from a base URL
2. Finds all internal links (same domain)
3. Extracts clean text from each page
4. Respects a maximum page limit (to avoid crawling massive sites)

### Important: Responsible Crawling
- We only crawl pages on the **same domain**
- We set a **page limit** to avoid overloading the server
- We add a **User-Agent** header to identify ourselves
- We respect **robots.txt** in production (simplified here for learning)

In [ ]:
class SimpleWebCrawler:
    """A simple web crawler that extracts text from pages on a single domain."""
    
    def __init__(self, base_url: str, max_pages: int = 20):
        self.base_url = base_url
        self.max_pages = max_pages
        self.domain = urlparse(base_url).netloc
        self.visited = set()
        self.pages = []  # List of {url, title, text}
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "FAQ-Bot-Crawler/1.0 (Educational Project)"
        })
    
    def _is_same_domain(self, url: str) -> bool:
        """Check if URL belongs to the same domain."""
        return urlparse(url).netloc == self.domain
    
    def _clean_url(self, url: str) -> str:
        """Remove fragments and normalize URL."""
        parsed = urlparse(url)
        return f"{parsed.scheme}://{parsed.netloc}{parsed.path}".rstrip("/")
    
    def _extract_text(self, soup: BeautifulSoup) -> str:
        """Extract clean text from HTML, removing scripts, styles, nav, etc."""
        # Remove non-content elements
        for tag in soup.find_all(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()
        
        # Get text and clean whitespace
        text = soup.get_text(separator="\n", strip=True)
        # Collapse multiple blank lines
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        return "\n".join(lines)
    
    def _find_links(self, soup: BeautifulSoup, current_url: str) -> list[str]:
        """Find all internal links on the page."""
        links = []
        for a_tag in soup.find_all("a", href=True):
            href = a_tag["href"]
            # Convert relative URLs to absolute
            full_url = urljoin(current_url, href)
            full_url = self._clean_url(full_url)
            
            # Only follow same-domain, non-visited links
            if (self._is_same_domain(full_url) 
                and full_url not in self.visited
                and not full_url.endswith((".pdf", ".png", ".jpg", ".zip"))):
                links.append(full_url)
        
        return list(set(links))  # Deduplicate
    
    def crawl(self) -> list[dict]:
        """Crawl the website starting from base_url."""
        to_visit = [self._clean_url(self.base_url)]
        
        while to_visit and len(self.visited) < self.max_pages:
            url = to_visit.pop(0)
            if url in self.visited:
                continue
            
            try:
                response = self.session.get(url, timeout=10)
                if response.status_code != 200:
                    continue
                if "text/html" not in response.headers.get("content-type", ""):
                    continue
                
                soup = BeautifulSoup(response.text, "html.parser")
                title = soup.title.string if soup.title else url
                text = self._extract_text(soup)
                
                if len(text) > 50:  # Skip near-empty pages
                    self.pages.append({
                        "url": url,
                        "title": title.strip() if title else url,
                        "text": text,
                    })
                    print(f"  ✓ [{len(self.pages)}] {title.strip()[:60] if title else url}")
                
                self.visited.add(url)
                
                # Find and queue new links
                new_links = self._find_links(soup, url)
                to_visit.extend(new_links)
                
            except Exception as e:
                print(f"  ✗ Error crawling {url}: {e}")
                self.visited.add(url)
        
        return self.pages


print("Crawler class defined!")

## Step 3 — Crawl a Website

Let's crawl a real website. We'll use Python's official documentation
(just a few pages as demo).

In [ ]:
# Crawl Python's FAQ page (small, good for demo)
TARGET_URL = "https://docs.python.org/3/faq/"

print(f"🕷️ Crawling: {TARGET_URL}")
print(f"   Max pages: 10\n")

crawler = SimpleWebCrawler(TARGET_URL, max_pages=10)
pages = crawler.crawl()

print(f"\n📊 Crawled {len(pages)} pages")
for page in pages:
    print(f"   {page['title'][:50]} — {len(page['text'])} chars")

## Step 4 — Convert to LangChain Documents and Chunk

In [ ]:
# Convert crawled pages to LangChain Document objects
documents = [
    Document(
        page_content=page["text"],
        metadata={"source": page["url"], "title": page["title"]},
    )
    for page in pages
]

print(f"Created {len(documents)} documents")

# Chunk the documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " "],
)

chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

# Preview a chunk
print(f"\n--- Sample Chunk ---")
print(f"Source: {chunks[0].metadata['source']}")
print(f"Content: {chunks[0].page_content[:200]}...")

## Step 5 — Create Vector Store and Retriever

In [ ]:
# Initialize embedding model
embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")

# Create vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="website_faq",
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

print(f"Vector store ready with {vectorstore._collection.count()} vectors")

## Step 6 — Build the FAQ Chain

In [ ]:
# Initialize LLM
llm = ChatOllama(model="qwen3.5:9b", temperature=0.3)

# FAQ-specific prompt
faq_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a helpful FAQ assistant for a website. Answer the user's question
based ONLY on the website content provided below.

Guidelines:
- Answer clearly and concisely
- If the answer spans multiple pages, synthesize the information
- Always mention which page/section the answer comes from
- If the information isn't in the provided content, say so

Website content:
{context}

User question: {question}

Answer:"""
)

# Build the QA chain
faq_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": faq_prompt},
)

print("FAQ chain ready!")

## Step 7 — Ask Questions About the Website!

In [ ]:
def ask_faq(question: str) -> None:
    """Ask a question about the crawled website."""
    print(f"❓ {question}")
    print("-" * 60)
    
    result = faq_chain.invoke({"query": question})
    
    print(f"\n💡 {result['result']}")
    
    # Show sources
    sources = set()
    for doc in result["source_documents"]:
        sources.add(doc.metadata.get("source", "unknown"))
    print(f"\n📎 Sources: {', '.join(sources)}")
    print("=" * 60 + "\n")


# Ask questions about Python FAQ
ask_faq("What is Python and what is it good for?")

In [ ]:
ask_faq("How does Python handle memory management?")

In [ ]:
ask_faq("What is the difference between a list and a tuple?")

## Step 8 — Complete Pipeline Function

Wrap everything so you can quickly build a FAQ bot for any website.

In [ ]:
def build_faq_bot(url: str, max_pages: int = 15):
    """
    Build a FAQ bot for any website.
    
    Args:
        url: The website URL to crawl
        max_pages: Maximum pages to crawl
    
    Returns:
        A function that answers questions about the website
    """
    # 1. Crawl
    print(f"🕷️ Crawling {url} (max {max_pages} pages)...")
    crawler = SimpleWebCrawler(url, max_pages=max_pages)
    pages = crawler.crawl()
    
    # 2. Chunk
    docs = [
        Document(page_content=p["text"], metadata={"source": p["url"], "title": p["title"]})
        for p in pages
    ]
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    chunks = splitter.split_documents(docs)
    print(f"📦 {len(chunks)} chunks from {len(pages)} pages")
    
    # 3. Embed and store
    vs = Chroma.from_documents(chunks, embeddings, collection_name="faq_bot")
    retriever = vs.as_retriever(search_kwargs={"k": 4})
    
    # 4. Build chain
    chain = RetrievalQA.from_chain_type(
        llm=llm, retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={"prompt": faq_prompt},
    )
    
    def ask(question: str) -> str:
        result = chain.invoke({"query": question})
        return result["result"]
    
    print("✅ FAQ bot ready!")
    return ask


# Usage:
# bot = build_faq_bot("https://example.com/docs")
# answer = bot("How do I get started?")
# print(answer)

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Web Crawling** | Automatically visits pages and follows internal links |
| **BeautifulSoup** | Parses HTML and extracts clean text content |
| **Same-domain filter** | Prevents the crawler from leaving the target website |
| **Document metadata** | Stores URL and title alongside text for citation |
| **RAG Pipeline** | Retrieve relevant chunks, then generate answers |
| **RetrievalQA** | LangChain's built-in retrieve-and-answer chain |

## 🔧 Customization Ideas

- **Respect robots.txt**: Use `robotparser` to check crawl permissions
- **Rate limiting**: Add delays between requests to be polite
- **Incremental updates**: Re-crawl only changed pages
- **Multi-site**: Support multiple websites in the same vector store
- **Chat memory**: Add conversation history for follow-up questions